# Results Visualisation
COMP6242 RNN's Revenge — Paradigm Comparison: Transformer vs minGRU vs Causal gMLP

This notebook generates publication-quality figures from the experiment results.
No GPU required — reads JSON/CSV summaries from each model's output directory.

**Figures:**
1. Copy task: Recall PPL vs sequence length
2. Induction task: Accuracy vs sequence length
3. Shakespeare: Val PPL vs context length
4. Wall clock time vs sequence length
5. Throughput (tok/s) vs sequence length
6. Training dynamics: Transformer copy-medium phase transition
7. Summary heatmap (pass/fail across all 27 experiments)

In [ ]:
import json, glob, csv, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.facecolor': 'white',
})

SAVE_DIR = 'figures'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Ready.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/DL-RNNs-Revenge'
os.chdir(BASE)
print('Working dir:', os.getcwd())

## Load Results

In [ ]:
# --- Transformer results ---
transformer = {}
for p in sorted(glob.glob('transformer_project/out/*/summary.json')):
    with open(p) as f:
        s = json.load(f)
    transformer[s['run_name']] = s

# --- gMLP results ---
gmlp = {}
for p in sorted(glob.glob('vibhansh-gMLP/results/gmlp_*.json')):
    with open(p) as f:
        s = json.load(f)
    name = os.path.basename(p).replace('.json', '')
    gmlp[name] = s

# --- minGRU results ---
mingru = {}
for p in sorted(glob.glob('minGRU/runs/*/metrics.csv')):
    run_name = p.split('/')[2]
    with open(p) as f:
        rows = list(csv.DictReader(f))
    if rows:
        best = min(rows, key=lambda r: float(r['val_loss']))
        mingru[run_name] = {
            'best_val_ppl': float(best['val_ppl']),
            'masked_ppl': float(best['masked_ppl']),
            'masked_acc': float(best['masked_acc']),
            'wall_clock_s': float(best['wall_clock_s']),
            'tokens_per_s': float(best['tokens_per_s']),
            'all_rows': rows,
        }

print(f'Transformer: {len(transformer)} runs')
print(f'gMLP: {len(gmlp)} runs')
print(f'minGRU: {len(mingru)} runs')

# Quick sanity
for name, s in transformer.items():
    print(f'  T: {name} -> val_ppl={s["best_val_ppl"]:.4f}')
for name, s in gmlp.items():
    print(f'  G: {name} -> val_ppl={s["best_val_ppl"]:.4f}')
for name, s in mingru.items():
    print(f'  M: {name} -> val_ppl={s["best_val_ppl"]:.4f}')

## Helper: Extract metrics by task

In [ ]:
# Map run names to canonical keys
LENGTHS = ['short', 'medium', 'long']
LENGTH_LABELS = {'short': 'Short', 'medium': 'Medium', 'long': 'Long'}
SHAK_LENGTHS = [256, 1024, 2048]

COLORS = {'Transformer': '#2196F3', 'minGRU': '#FF9800', 'gMLP': '#4CAF50'}
MARKERS = {'Transformer': 'o', 'minGRU': 's', 'gMLP': '^'}


def get_transformer_copy():
    recall_ppl = []
    for length in LENGTHS:
        key = f'lrcopy-{length}-s42-d05'
        recall_ppl.append(transformer[key]['best_recall_ppl'])
    return recall_ppl


def get_gmlp_copy():
    recall_ppl = []
    for length in LENGTHS:
        key = f'gmlp_copy_{length}_42'
        fr = gmlp[key].get('final_results', {})
        recall_ppl.append(fr.get('discriminating_ppl', float('nan')))
    return recall_ppl


def get_mingru_copy():
    recall_ppl = []
    for length in LENGTHS:
        for name, data in mingru.items():
            if f'longcopy_{length}' in name:
                recall_ppl.append(data['masked_ppl'])
                break
    return recall_ppl


def get_transformer_induction():
    acc = []
    for length in LENGTHS:
        key = f'induction-{length}-s42-d05'
        acc.append(transformer[key]['best_induction5_accuracy'])
    return acc


def get_gmlp_induction():
    acc = []
    for length in LENGTHS:
        key = f'gmlp_induction_{length}_42'
        fr = gmlp[key].get('final_results', {})
        a = fr.get('induction5_accuracy', fr.get('accuracy', float('nan')))
        acc.append(a)
    return acc


def get_mingru_induction():
    acc = []
    for length in LENGTHS:
        for name, data in mingru.items():
            if f'induction_{length}' in name:
                acc.append(data['masked_acc'])
                break
    return acc


def get_shakespeare_ppl():
    t_ppl, g_ppl, m_ppl = [], [], []
    for bs in SHAK_LENGTHS:
        t_ppl.append(transformer[f'tshake-{bs}-s42-d05']['best_val_ppl'])
        g_ppl.append(gmlp[f'gmlp_shakespeare_{bs}_42']['best_val_ppl'])
        for name, data in mingru.items():
            if f'tinyshakespeare_{bs}' in name:
                m_ppl.append(data['best_val_ppl'])
                break
    return t_ppl, g_ppl, m_ppl


print('Helpers defined.')

## Plot 1: Copy Task — Recall PPL vs Sequence Length

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

x = np.arange(len(LENGTHS))
width = 0.25

t_copy = get_transformer_copy()
g_copy = get_gmlp_copy()
m_copy = get_mingru_copy()

bars1 = ax.bar(x - width, t_copy, width, label='Transformer', color=COLORS['Transformer'], edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x, g_copy, width, label='gMLP', color=COLORS['gMLP'], edgecolor='white', linewidth=0.5)
bars3 = ax.bar(x + width, m_copy, width, label='minGRU', color=COLORS['minGRU'], edgecolor='white', linewidth=0.5)

ax.axhline(y=26, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Random baseline (~26)')
ax.axhline(y=1.0, color='green', linestyle=':', linewidth=1, alpha=0.7, label='Perfect recall (1.0)')

ax.set_xlabel('Sequence Length')
ax.set_ylabel('Recall PPL (lower = better)')
ax.set_title('Long-Range Copy: Discriminating Perplexity')
ax.set_xticks(x)
ax.set_xticklabels(['Short (128)', 'Medium (528)', 'Long (2048)'])
ax.set_ylim(0, 30)
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)

# Annotate key values
for bars, vals in [(bars1, t_copy), (bars2, g_copy), (bars3, m_copy)]:
    for bar, val in zip(bars, vals):
        if val < 5:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/01_copy_recall_ppl.png', dpi=200, bbox_inches='tight')
plt.show()

## Plot 2: Induction Task — Accuracy vs Sequence Length

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

x = np.arange(len(LENGTHS))
width = 0.25

t_ind = get_transformer_induction()
g_ind = get_gmlp_induction()
m_ind = get_mingru_induction()

bars1 = ax.bar(x - width, t_ind, width, label='Transformer', color=COLORS['Transformer'], edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x, g_ind, width, label='gMLP', color=COLORS['gMLP'], edgecolor='white', linewidth=0.5)
bars3 = ax.bar(x + width, m_ind, width, label='minGRU', color=COLORS['minGRU'], edgecolor='white', linewidth=0.5)

ax.axhline(y=1/27, color='gray', linestyle='--', linewidth=1, alpha=0.7, label=f'Random chance (1/27 ≈ {1/27:.3f})')

ax.set_xlabel('Sequence Length')
ax.set_ylabel('Induction Accuracy (P[4] prediction)')
ax.set_title('Induction Head: 5th Character Prediction Accuracy')
ax.set_xticks(x)
ax.set_xticklabels(['Short (110)', 'Medium (410)', 'Long (2010)'])
ax.set_ylim(0, 1.1)
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

# Annotate
for bars, vals in [(bars1, t_ind), (bars2, g_ind), (bars3, m_ind)]:
    for bar, val in zip(bars, vals):
        if val > 0.1:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/02_induction_accuracy.png', dpi=200, bbox_inches='tight')
plt.show()

## Plot 3: Shakespeare — Val PPL vs Context Length

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

t_ppl, g_ppl, m_ppl = get_shakespeare_ppl()

for name, ppl_vals, color, marker in [
    ('Transformer', t_ppl, COLORS['Transformer'], MARKERS['Transformer']),
    ('gMLP', g_ppl, COLORS['gMLP'], MARKERS['gMLP']),
    ('minGRU', m_ppl, COLORS['minGRU'], MARKERS['minGRU']),
]:
    ax.plot(SHAK_LENGTHS, ppl_vals, marker=marker, label=name, color=color,
            linewidth=2, markersize=8)

ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Validation Perplexity')
ax.set_title('TinyShakespeare: Language Modelling Performance')
ax.set_xscale('log', base=2)
ax.set_xticks(SHAK_LENGTHS)
ax.set_xticklabels(['256', '1024', '2048'])
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(3, 5.5)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03_shakespeare_ppl.png', dpi=200, bbox_inches='tight')
plt.show()

## Plot 4: Wall Clock Time vs Sequence Length

In [ ]:
# Collect wall clock from summaries
# Transformer: wall_clock_sec
t_time_shak = [transformer[f'tshake-{bs}-s42-d05'].get('wall_clock_sec', float('nan')) for bs in SHAK_LENGTHS]
t_time_copy = [transformer[f'lrcopy-{l}-s42-d05'].get('wall_clock_sec', float('nan')) for l in LENGTHS]

# gMLP: training_time_sec
g_time_shak = [gmlp[f'gmlp_shakespeare_{bs}_42'].get('training_time_sec', float('nan')) for bs in SHAK_LENGTHS]
g_time_copy = [gmlp[f'gmlp_copy_{l}_42'].get('training_time_sec', float('nan')) for l in LENGTHS]

# minGRU: wall_clock_s from last row of metrics
m_time_shak = []
for bs in SHAK_LENGTHS:
    for name, data in mingru.items():
        if f'tinyshakespeare_{bs}' in name:
            last_row = data['all_rows'][-1]
            m_time_shak.append(float(last_row['wall_clock_s']))
            break

m_time_copy = []
for length in LENGTHS:
    for name, data in mingru.items():
        if f'longcopy_{length}' in name:
            last_row = data['all_rows'][-1]
            m_time_copy.append(float(last_row['wall_clock_s']))
            break

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Shakespeare
for name, times, color, marker in [
    ('Transformer', t_time_shak, COLORS['Transformer'], MARKERS['Transformer']),
    ('gMLP', g_time_shak, COLORS['gMLP'], MARKERS['gMLP']),
    ('minGRU', m_time_shak, COLORS['minGRU'], MARKERS['minGRU']),
]:
    ax1.plot(SHAK_LENGTHS, times, marker=marker, label=name, color=color,
             linewidth=2, markersize=8)

ax1.set_xlabel('Context Length')
ax1.set_ylabel('Wall Clock (seconds)')
ax1.set_title('Shakespeare: Training Time (5K steps)')
ax1.set_xscale('log', base=2)
ax1.set_xticks(SHAK_LENGTHS)
ax1.set_xticklabels(['256', '1024', '2048'])
ax1.legend()
ax1.grid(alpha=0.3)

# Copy
copy_seq_lens = [128, 528, 2048]
for name, times, color, marker in [
    ('Transformer', t_time_copy, COLORS['Transformer'], MARKERS['Transformer']),
    ('gMLP', g_time_copy, COLORS['gMLP'], MARKERS['gMLP']),
    ('minGRU', m_time_copy, COLORS['minGRU'], MARKERS['minGRU']),
]:
    valid = [(s, t) for s, t in zip(copy_seq_lens, times) if not np.isnan(t)]
    if valid:
        ax2.plot([v[0] for v in valid], [v[1] for v in valid],
                 marker=marker, label=name, color=color, linewidth=2, markersize=8)

ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Wall Clock (seconds)')
ax2.set_title('Copy Task: Training Time (5K steps)')
ax2.set_xscale('log', base=2)
ax2.set_xticks(copy_seq_lens)
ax2.set_xticklabels(['128', '528', '2048'])
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_wall_clock.png', dpi=200, bbox_inches='tight')
plt.show()

## Plot 5: Throughput (tok/s) vs Sequence Length

In [ ]:
# Transformer throughput: throughput_tok_per_sec
t_tps_shak = [transformer[f'tshake-{bs}-s42-d05'].get('throughput_tok_per_sec', float('nan')) for bs in SHAK_LENGTHS]
t_tps_copy = [transformer[f'lrcopy-{l}-s42-d05'].get('throughput_tok_per_sec', float('nan')) for l in LENGTHS]

# minGRU throughput from metrics (best row tokens_per_s)
m_tps_shak = []
for bs in SHAK_LENGTHS:
    for name, data in mingru.items():
        if f'tinyshakespeare_{bs}' in name:
            m_tps_shak.append(data['tokens_per_s'])
            break

m_tps_copy = []
for length in LENGTHS:
    for name, data in mingru.items():
        if f'longcopy_{length}' in name:
            m_tps_copy.append(data['tokens_per_s'])
            break

# gMLP: compute from total_tokens / training_time_sec
g_tps_shak = []
for bs in SHAK_LENGTHS:
    s = gmlp[f'gmlp_shakespeare_{bs}_42']
    wc = s.get('training_time_sec', float('nan'))
    batch = s.get('config', {}).get('batch_size', 64 if bs <= 1024 else 32)
    total_tok = s.get('max_steps', 5000) * batch * bs
    g_tps_shak.append(total_tok / wc if wc > 0 else float('nan'))

g_tps_copy = []
copy_block_sizes = [144, 544, 2048]
for i, length in enumerate(LENGTHS):
    s = gmlp[f'gmlp_copy_{length}_42']
    wc = s.get('training_time_sec', float('nan'))
    batch = s.get('config', {}).get('batch_size', 64 if i < 2 else 32)
    total_tok = s.get('max_steps', 5000) * batch * copy_block_sizes[i]
    g_tps_copy.append(total_tok / wc if wc > 0 else float('nan'))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

for name, tps, color, marker in [
    ('Transformer', t_tps_shak, COLORS['Transformer'], MARKERS['Transformer']),
    ('gMLP', g_tps_shak, COLORS['gMLP'], MARKERS['gMLP']),
    ('minGRU', m_tps_shak, COLORS['minGRU'], MARKERS['minGRU']),
]:
    valid = [(s, t) for s, t in zip(SHAK_LENGTHS, tps) if not np.isnan(t)]
    if valid:
        ax1.plot([v[0] for v in valid], [v[1]/1e6 for v in valid],
                 marker=marker, label=name, color=color, linewidth=2, markersize=8)

ax1.set_xlabel('Context Length')
ax1.set_ylabel('Throughput (M tok/s)')
ax1.set_title('Shakespeare: Training Throughput')
ax1.set_xscale('log', base=2)
ax1.set_xticks(SHAK_LENGTHS)
ax1.set_xticklabels(['256', '1024', '2048'])
ax1.legend()
ax1.grid(alpha=0.3)

copy_seq_lens = [128, 528, 2048]
for name, tps, color, marker in [
    ('Transformer', t_tps_copy, COLORS['Transformer'], MARKERS['Transformer']),
    ('gMLP', g_tps_copy, COLORS['gMLP'], MARKERS['gMLP']),
    ('minGRU', m_tps_copy, COLORS['minGRU'], MARKERS['minGRU']),
]:
    valid = [(s, t) for s, t in zip(copy_seq_lens, tps) if not np.isnan(t)]
    if valid:
        ax2.plot([v[0] for v in valid], [v[1]/1e6 for v in valid],
                 marker=marker, label=name, color=color, linewidth=2, markersize=8)

ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Throughput (M tok/s)')
ax2.set_title('Copy Task: Training Throughput')
ax2.set_xscale('log', base=2)
ax2.set_xticks(copy_seq_lens)
ax2.set_xticklabels(['128', '528', '2048'])
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/05_throughput.png', dpi=200, bbox_inches='tight')
plt.show()

## Plot 6: Training Dynamics — Transformer Copy-Medium Phase Transition

In [ ]:
# Try to read from transformer summary eval_history first
s = transformer.get('lrcopy-medium-s42-d05', {})
eval_history = s.get('eval_history', [])

if eval_history:
    steps_medium = [e['step'] for e in eval_history]
    recall_ppl_medium = [e.get('recall_ppl', float('nan')) for e in eval_history]
    print(f'Loaded {len(steps_medium)} eval points from summary.json')
else:
    # Fallback: hardcoded from validated notebook output
    steps_medium = [0, 250, 500, 750, 1000, 1250, 1500, 1750, 2000, 2250, 2500,
                    2750, 3000, 3250, 3500, 3750, 4000, 4250, 4500, 4750, 5000]
    recall_ppl_medium = [56.441, 26.325, 26.171, 26.222, 26.248, 26.222, 26.120,
                         26.145, 26.043, 26.094, 26.120, 26.043, 14.326, 6.108,
                         2.567, 1.461, 1.148, 1.070, 1.044, 1.032, 1.025]
    print('Using hardcoded eval points from notebook logs')

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(steps_medium, recall_ppl_medium, color=COLORS['Transformer'],
        linewidth=2.5, marker='o', markersize=5)

# Highlight phase transition region
ax.axvspan(2750, 4000, alpha=0.1, color='red', label='Phase transition')
ax.axhline(y=26, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Random baseline')
ax.axhline(y=1.0, color='green', linestyle=':', linewidth=1, alpha=0.7, label='Perfect recall')

ax.set_xlabel('Training Step')
ax.set_ylabel('Recall PPL')
ax.set_title('Transformer Copy-Medium (T=528): Phase Transition in Recall')
ax.legend(loc='right')
ax.grid(alpha=0.3)
ax.set_ylim(0, 30)

# Annotate the drop
ax.annotate('Attention heads\nsnap into copy pattern',
            xy=(3250, 6.1), xytext=(3800, 15),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=9, color='red', ha='center')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/06_phase_transition.png', dpi=200, bbox_inches='tight')
plt.show()

## Plot 7: Summary Heatmap

In [ ]:
# Build the 3x9 results matrix
# Rows: models (Transformer, gMLP, minGRU)
# Cols: task-length combinations

tasks = [
    ('Shak\n256', 'shakespeare', 256),
    ('Shak\n1024', 'shakespeare', 1024),
    ('Shak\n2048', 'shakespeare', 2048),
    ('Copy\nShort', 'copy', 'short'),
    ('Copy\nMed', 'copy', 'medium'),
    ('Copy\nLong', 'copy', 'long'),
    ('Ind\nShort', 'induction', 'short'),
    ('Ind\nMed', 'induction', 'medium'),
    ('Ind\nLong', 'induction', 'long'),
]

models = ['Transformer', 'gMLP', 'minGRU']

# For copy/induction: "pass" = recall_ppl < 2 or accuracy > 0.5
# For shakespeare: always "pass" (LM task, no binary pass/fail)
# Score: 0=fail, 0.5=partial, 1=pass

def score_copy(recall_ppl):
    if recall_ppl < 1.1: return 1.0
    if recall_ppl < 5: return 0.5
    return 0.0

def score_induction(acc):
    if acc > 0.9: return 1.0
    if acc > 0.3: return 0.5
    return 0.0

def score_shakespeare(ppl):
    if ppl < 4.0: return 1.0
    if ppl < 5.0: return 0.7
    return 0.4

# Get all scores
t_copy = get_transformer_copy()
g_copy = get_gmlp_copy()
m_copy = get_mingru_copy()
t_ind = get_transformer_induction()
g_ind = get_gmlp_induction()
m_ind = get_mingru_induction()
t_shak, g_shak, m_shak = get_shakespeare_ppl()

scores = np.zeros((3, 9))
annotations = [[''] * 9 for _ in range(3)]

# Transformer row
for i, ppl in enumerate(t_shak):
    scores[0, i] = score_shakespeare(ppl)
    annotations[0][i] = f'{ppl:.2f}'
for i, rppl in enumerate(t_copy):
    scores[0, 3+i] = score_copy(rppl)
    annotations[0][3+i] = f'{rppl:.2f}'
for i, acc in enumerate(t_ind):
    scores[0, 6+i] = score_induction(acc)
    annotations[0][6+i] = f'{acc:.3f}'

# gMLP row
for i, ppl in enumerate(g_shak):
    scores[1, i] = score_shakespeare(ppl)
    annotations[1][i] = f'{ppl:.2f}'
for i, rppl in enumerate(g_copy):
    scores[1, 3+i] = score_copy(rppl)
    annotations[1][3+i] = f'{rppl:.2f}'
for i, acc in enumerate(g_ind):
    scores[1, 6+i] = score_induction(acc)
    annotations[1][6+i] = f'{acc:.3f}'

# minGRU row
for i, ppl in enumerate(m_shak):
    scores[2, i] = score_shakespeare(ppl)
    annotations[2][i] = f'{ppl:.2f}'
for i, rppl in enumerate(m_copy):
    scores[2, 3+i] = score_copy(rppl)
    annotations[2][3+i] = f'{rppl:.2f}'
for i, acc in enumerate(m_ind):
    scores[2, 6+i] = score_induction(acc)
    annotations[2][6+i] = f'{acc:.3f}'

# Custom colormap: red -> yellow -> green
cmap = LinearSegmentedColormap.from_list('rg', ['#ef5350', '#ffee58', '#66bb6a'])

fig, ax = plt.subplots(figsize=(12, 3.5))

im = ax.imshow(scores, cmap=cmap, vmin=0, vmax=1, aspect='auto')

# Labels
ax.set_xticks(range(9))
ax.set_xticklabels([t[0] for t in tasks], fontsize=8)
ax.set_yticks(range(3))
ax.set_yticklabels(models)

# Annotate cells
for i in range(3):
    for j in range(9):
        color = 'white' if scores[i, j] < 0.3 else 'black'
        ax.text(j, i, annotations[i][j], ha='center', va='center',
                fontsize=8, fontweight='bold', color=color)

# Section dividers
ax.axvline(x=2.5, color='white', linewidth=2)
ax.axvline(x=5.5, color='white', linewidth=2)

# Section headers
ax.text(1, -0.7, 'Shakespeare (Val PPL)', ha='center', fontsize=9, fontweight='bold')
ax.text(4, -0.7, 'Copy (Recall PPL)', ha='center', fontsize=9, fontweight='bold')
ax.text(7, -0.7, 'Induction (Accuracy)', ha='center', fontsize=9, fontweight='bold')

ax.set_title('Experiment Summary: All Models × All Tasks × All Lengths', pad=25)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/07_summary_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nAll figures saved to figures/ directory.')